In [1]:
# Importing the necessary libraries
from pyspark import SparkContext
from pyspark.sql import SparkSession

# Creating a SparkSession
spark = SparkSession.builder.master('local').getOrCreate()

In [2]:
## reading in pyspark df
spark_df = spark.read.csv('./forestfires.csv', header=True, inferSchema=True)

## observing the datatype of df
type(spark_df)

pyspark.sql.dataframe.DataFrame

In [3]:
spark_df.head()

Row(X=7, Y=5, month='mar', day='fri', FFMC=86.2, DMC=26.2, DC=94.3, ISI=5.1, temp=8.2, RH=51, wind=6.7, rain=0.0, area=0.0)

In [4]:
spark_df.columns

['X',
 'Y',
 'month',
 'day',
 'FFMC',
 'DMC',
 'DC',
 'ISI',
 'temp',
 'RH',
 'wind',
 'rain',
 'area']

In [5]:
spark_df[['month', 'day', 'rain']].show()

+-----+---+----+
|month|day|rain|
+-----+---+----+
|  mar|fri| 0.0|
|  oct|tue| 0.0|
|  oct|sat| 0.0|
|  mar|fri| 0.2|
|  mar|sun| 0.0|
|  aug|sun| 0.0|
|  aug|mon| 0.0|
|  aug|mon| 0.0|
|  sep|tue| 0.0|
|  sep|sat| 0.0|
|  sep|sat| 0.0|
|  sep|sat| 0.0|
|  aug|fri| 0.0|
|  sep|mon| 0.0|
|  sep|wed| 0.0|
|  sep|fri| 0.0|
|  mar|sat| 0.0|
|  oct|mon| 0.0|
|  mar|wed| 0.0|
|  apr|sat| 0.0|
+-----+---+----+
only showing top 20 rows



In [6]:
spark_df.select('rain')

DataFrame[rain: double]

In [7]:
spark_df['rain']

Column<'rain'>

In [8]:
spark_df.dtypes

[('X', 'int'),
 ('Y', 'int'),
 ('month', 'string'),
 ('day', 'string'),
 ('FFMC', 'double'),
 ('DMC', 'double'),
 ('DC', 'double'),
 ('ISI', 'double'),
 ('temp', 'double'),
 ('RH', 'int'),
 ('wind', 'double'),
 ('rain', 'double'),
 ('area', 'double')]

In [9]:
spark_df_months = spark_df.groupBy('month').agg({'area': 'mean'})
spark_df_months

DataFrame[month: string, avg(area): double]

In [11]:
spark_df_months.collect()

[Row(month='jun', avg(area)=5.841176470588234),
 Row(month='aug', avg(area)=12.489076086956521),
 Row(month='may', avg(area)=19.24),
 Row(month='feb', avg(area)=6.275),
 Row(month='sep', avg(area)=17.942616279069753),
 Row(month='mar', avg(area)=4.356666666666667),
 Row(month='oct', avg(area)=6.638),
 Row(month='jul', avg(area)=14.3696875),
 Row(month='nov', avg(area)=0.0),
 Row(month='apr', avg(area)=8.891111111111112),
 Row(month='dec', avg(area)=13.33),
 Row(month='jan', avg(area)=0.0)]

In [12]:
no_rain = spark_df.filter(spark_df.rain == 0.0)
some_rain = spark_df.filter(spark_df.rain > 0.0)

In [13]:
from pyspark.sql.functions import mean

print("no rain fire area:", no_rain.select(mean('area')).show(), "\n")
print("some rain fire area:", some_rain.select(mean('area')).show(), "\n")

+------------------+
|         avg(area)|
+------------------+
|13.023693516699408|
+------------------+

no rain fire area: None 

+---------+
|avg(area)|
+---------+
|  1.62375|
+---------+

some rain fire area: None 



In [14]:
summer_months = spark_df.filter(spark_df['month'].isin(['jun', 'jul', 'aug']))
winter_months = spark_df.filter(spark_df['month'].isin(['dec', 'jan', 'feb']))

print('summer months fire area', summer_months.select(mean('area')).show())
print('winter months fire areas', winter_months.select(mean('area')).show())

+------------------+
|         avg(area)|
+------------------+
|12.262317596566525|
+------------------+

summer months fire area None
+-----------------+
|        avg(area)|
+-----------------+
|7.918387096774193|
+-----------------+

winter months fire areas None


In [15]:
from pyspark.sql import functions as F

# bin months by season
df_months_binned = spark_df.withColumn('month',
                                         F.when(spark_df['month'] == 'jun', 'Summer') \
                                         .when(spark_df['month'] == 'jul', 'Summer') \
                                         .when(spark_df['month'] == 'aug', 'Summer') \
                                         .when(spark_df['month'] == 'jan', 'Winter') \
                                         .when(spark_df['month'] == 'feb', 'Winter') \
                                         .when(spark_df['month'] == 'dec', 'Winter') \
                                         .otherwise('Spring/Fall')
                                        )

# Select both 'month' and 'area' before grouping and aggregating
result = df_months_binned.select('month', 'area').groupBy('month').agg({'area': 'mean'}).distinct()
result.show() # To display the results

+-----------+------------------+
|      month|         avg(area)|
+-----------+------------------+
|     Summer|12.262317596566525|
|Spring/Fall|13.989960474308296|
|     Winter| 7.918387096774193|
+-----------+------------------+

